In [2]:
from azureml.core import Workspace, Experiment
subscription_id= "f5091c60-1c3c-430f-8d81-d802f6bf2414"
resource_group= "aml-quickstarts-267237"
workspace_name= "quick-starts-ws-267237"

#ws = Workspace.from_config()
ws = Workspace.get(name=workspace_name, subscription_id=subscription_id, resource_group=resource_group)
exp = Experiment(workspace=ws, name="udacity-project")

print('Workspace name: ' + ws.name, 
      'Azure region: ' + ws.location, 
      'Subscription id: ' + ws.subscription_id, 
      'Resource group: ' + ws.resource_group, sep = '\n')

run = exp.start_logging()

StatementMeta(6c912745-d107-4a12-86f9-8f44b6841ca6, 1, 7, Finished, Available, Finished)

Performing interactive authentication. Please follow the instructions on the terminal.
To sign in, use a web browser to open the page https://microsoft.com/devicelogin and enter the code BBLHKJHFP to authenticate.
Interactive authentication successfully completed.
Workspace name: quick-starts-ws-267237
Azure region: eastus2
Subscription id: f5091c60-1c3c-430f-8d81-d802f6bf2414
Resource group: aml-quickstarts-267237


In [3]:
from azureml.core.compute import ComputeTarget, AmlCompute
cluster_name = "project-1"
vm_size="Standard_D2_V2"
compute_config=AmlCompute.provisioning_configuration(vm_size=vm_size,max_nodes=4)
# TODO: Create compute cluster
# Use vm_size = "Standard_D2_V2" in your provisioning configuration.
# max_nodes should be no greater than 4.
### YOUR CODE HERE ###
from azureml.exceptions import ComputeTargetException
try:
    compute_target=ComputeTarget(workspace=ws,name=cluster_name)
    print('Found existing compute cluster:',cluster_name)
except ComputeTargetException:
    compute_target=ComputeTarget.create(workspace=ws,name=cluster_name,provisioning_configuration=compute_config)
    compute_target.wait_for_completion(show_output=True)


StatementMeta(6c912745-d107-4a12-86f9-8f44b6841ca6, 1, 8, Finished, Available, Finished)

Found existing compute cluster: project-1


In [6]:
!pip install azureml-widgets -q
!pip install azureml-train-automl-runtime==1.57.0

StatementMeta(6c912745-d107-4a12-86f9-8f44b6841ca6, 1, 11, Finished, Available, Finished)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
azureml-opendatasets 1.47.0 requires azureml-core~=1.47.0, but you have azureml-core 1.57.0 which is incompatible.
azureml-opendatasets 1.47.0 requires azureml-telemetry~=1.47.0, but you have azureml-telemetry 1.57.0 which is incompatible.
azureml-mlflow 1.47.0 requires azure-storage-blob<=12.13.0,>=12.5.0, but you have azure-storage-blob 12.14.1 which is incompatible.
azure-identity 1.7.0 requires msal-extensions~=0.3.0, but you have msal-extensions 1.0.0 which is incompatible.
  Using cached azureml_train_automl_runtime-1.57.0-py3-none-any.whl (420 kB)
  Using cached boto3-1.20.19-py3-none-any.whl (131 kB)
  Using cached azureml_mlflow-1.57.0.post1-py3-none-any.whl (1.0 MB)
  Using cached onnxruntime-1.17.3-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (6.8 MB)
  Using cached onnxmltools-1.11.2-py2

In [7]:
from azureml.widgets import RunDetails
from azureml.train.sklearn import SKLearn
from azureml.train.hyperdrive.run import PrimaryMetricGoal
from azureml.train.hyperdrive.policy import BanditPolicy
from azureml.train.hyperdrive.sampling import RandomParameterSampling
from azureml.train.hyperdrive.runconfig import HyperDriveConfig
from azureml.train.hyperdrive.parameter_expressions import choice, uniform
from azureml.core import Environment, ScriptRunConfig
import os

# Specify parameter sampler
ps = RandomParameterSampling(parameter_space={'--C':uniform(0.5,1.5),'--max_iter':choice(16,32,64,128)})

# Specify a Policy
policy = BanditPolicy(slack_factor=0.2)

if "training" not in os.listdir():
    os.mkdir("./training")

# Setup environment for your training run
sklearn_env = Environment.from_conda_specification(
    name='sklearn-env', file_path='Users/odl_user_267237/conda_dependencies.yml')

# Create a ScriptRunConfig Object to specify the configuration details of your training job
src = ScriptRunConfig(
    source_directory='.',
    script='Users/odl_user_267237/train.py',
    environment=sklearn_env,compute_target=compute_target)

# Create a HyperDriveConfig using the src object, hyperparameter sampler, and policy.
hyperdrive_config = HyperDriveConfig(
    hyperparameter_sampling=ps,
    primary_metric_goal=PrimaryMetricGoal.MAXIMIZE,
    primary_metric_name='Accuracy',
    policy=policy,
    run_config=src,
    max_total_runs=15,
    max_concurrent_runs=4)

StatementMeta(6c912745-d107-4a12-86f9-8f44b6841ca6, 1, 12, Finished, Available, Finished)

In [8]:
# Submit your hyperdrive run to the experiment and show run details with the widget.

### YOUR CODE HERE ###
exp_run=exp.submit(hyperdrive_config)
RunDetails(exp_run).show()
exp_run.wait_for_completion(show_output=True)

StatementMeta(6c912745-d107-4a12-86f9-8f44b6841ca6, 1, 13, Finished, Available, Finished)

Failed to load image Python extension: libc10_cuda.so: cannot open shared object file: No such file or directory


Initializing logging file for interpret-community


_HyperDriveWidget(widget_settings={'childWidgetDisplay': 'popup', 'send_telemetry': False, 'log_level': 'INFO'…

RunId: HD_0b1a94d0-714d-4f1a-9d52-2a2652eba71a
Web View: https://ml.azure.com/runs/HD_0b1a94d0-714d-4f1a-9d52-2a2652eba71a?wsid=/subscriptions/f5091c60-1c3c-430f-8d81-d802f6bf2414/resourcegroups/aml-quickstarts-267237/workspaces/quick-starts-ws-267237&tid=660b3398-b80e-49d2-bc5b-ac1dc93b5254

Streaming azureml-logs/hyperdrive.txt

[2024-09-03T10:02:19.617499][GENERATOR][INFO]Trying to sample '4' jobs from the hyperparameter space
[2024-09-03T10:02:20.1121917Z][SCHEDULER][INFO]Scheduling job, id='HD_0b1a94d0-714d-4f1a-9d52-2a2652eba71a_0' 
[2024-09-03T10:02:20.2625557Z][SCHEDULER][INFO]Scheduling job, id='HD_0b1a94d0-714d-4f1a-9d52-2a2652eba71a_1' 
[2024-09-03T10:02:20.3810676Z][SCHEDULER][INFO]Successfully scheduled a job. Id='HD_0b1a94d0-714d-4f1a-9d52-2a2652eba71a_0' 
[2024-09-03T10:02:20.3795890Z][SCHEDULER][INFO]Scheduling job, id='HD_0b1a94d0-714d-4f1a-9d52-2a2652eba71a_2' 
[2024-09-03T10:02:20.4767582Z][SCHEDULER][INFO]Scheduling job, id='HD_0b1a94d0-714d-4f1a-9d52-2a2652eba71a_3

{'runId': 'HD_0b1a94d0-714d-4f1a-9d52-2a2652eba71a',
 'target': 'project-1',
 'status': 'Completed',
 'startTimeUtc': '2024-09-03T10:02:18.429879Z',
 'endTimeUtc': '2024-09-03T10:09:23.444489Z',
 'services': {},
 'properties': {'primary_metric_config': '{"name":"Accuracy","goal":"maximize"}',
  'resume_from': 'null',
  'runTemplate': 'HyperDrive',
  'azureml.runsource': 'hyperdrive',
  'platform': 'AML',
  'ContentSnapshotId': 'afa87780-688a-486b-87c0-5b37843e0ced',
  'user_agent': 'python/3.10.6 (Linux-4.15.0-1178-azure-x86_64-with-glibc2.27) msrest/0.6.21 Hyperdrive.Service/1.0.0 Hyperdrive.SDK/core.1.47.0',
  'space_size': 'infinite_space_size',
  'best_child_run_id': 'HD_0b1a94d0-714d-4f1a-9d52-2a2652eba71a_12',
  'score': '0.9083459787556905',
  'best_metric_status': 'Succeeded',
  'best_data_container_id': 'dcid.HD_0b1a94d0-714d-4f1a-9d52-2a2652eba71a_12'},
 'inputDatasets': [],
 'outputDatasets': [],
 'runDefinition': {'configuration': None,
  'attribution': None,
  'telemetryVa

In [9]:
import joblib
# Get your best run and save the model from that run.

### YOUR CODE HERE ###
best_run=exp_run.get_best_run_by_primary_metric()
print(best_run)

StatementMeta(6c912745-d107-4a12-86f9-8f44b6841ca6, 1, 14, Finished, Available, Finished)

Run(Experiment: udacity-project,
Id: HD_0b1a94d0-714d-4f1a-9d52-2a2652eba71a_12,
Type: azureml.scriptrun,
Status: Completed)


In [11]:
#print(best_run.get_details())
print(best_run.get_metrics())
print(best_run.get_file_names())

StatementMeta(6c912745-d107-4a12-86f9-8f44b6841ca6, 1, 16, Finished, Available, Finished)

{'Max iterations:': 64, 'Regularization Strength:': 1.0699733452615976, 'Accuracy': 0.9083459787556905}
['logs/azureml/dataprep/0/backgroundProcess.log', 'logs/azureml/dataprep/0/backgroundProcess_Telemetry.log', 'logs/azureml/dataprep/0/rslex.log.2024-09-03-10', 'system_logs/cs_capability/cs-capability.log', 'system_logs/hosttools_capability/hosttools-capability.log', 'system_logs/lifecycler/execution-wrapper.log', 'system_logs/lifecycler/lifecycler.log', 'system_logs/metrics_capability/metrics-capability.log', 'system_logs/snapshot_capability/snapshot-capability.log', 'user_logs/std_log.txt']


In [13]:
joblib.dump(best_run.get_metrics(),'best_run.json')

StatementMeta(6c912745-d107-4a12-86f9-8f44b6841ca6, 1, 18, Finished, Available, Finished)

['best_run.json']

In [18]:
from azureml.data.dataset_factory import TabularDatasetFactory

# Create TabularDataset using TabularDatasetFactory
# Data is available at: 
# "https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv"

### YOUR CODE HERE ###
ds = TabularDatasetFactory.from_delimited_files(path="https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv")

StatementMeta(6c912745-d107-4a12-86f9-8f44b6841ca6, 1, 23, Finished, Available, Finished)

In [19]:
from Users.odl_user_267237.train import clean_data


# Use the clean_data function to clean your data.
x, y = clean_data(ds)

StatementMeta(6c912745-d107-4a12-86f9-8f44b6841ca6, 1, 24, Finished, Available, Finished)

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/trusted-service-user/cluster-env/env/lib/python3.10/site-packages/IPython/core/interactiveshell.py", line 3433, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_24459/1453029740.py", line 5, in <module>
    x, y = clean_data(ds)
  File "/synfs/notebook/1/aml_notebook_mount/Users/odl_user_267237/train.py", line 19, in clean_data
    x_df = data.to_pandas_dataframe().dropna()
  File "/home/trusted-service-user/cluster-env/env/lib/python3.10/site-packages/azureml/data/_loggerfactory.py", line 132, in wrapper
  File "/home/trusted-service-user/cluster-env/env/lib/python3.10/site-packages/azureml/data/tabular_dataset.py", line 168, in to_pandas_dataframe
    dataflow = get_dataflow_for_execution(self._dataflow, 'to_pandas_dataframe', 'TabularDataset')
  File "/home/trusted-service-user/cluster-env/env/lib/python3.10/site-packages/azureml/data/_dataprep_helper.py", line 76, in get_dataflow_for_execution


In [ ]:
from azureml.train.automl import AutoMLConfig

# Set parameters for AutoMLConfig
# NOTE: DO NOT CHANGE THE experiment_timeout_minutes PARAMETER OR YOUR INSTANCE WILL TIME OUT.
# If you wish to run the experiment longer, you will need to run this notebook in your own
# Azure tenant, which will incur personal costs.
automl_config = AutoMLConfig(
    experiment_timeout_minutes=30,
    task=,
    primary_metric=,
    training_data=,
    label_column_name=,
    n_cross_validations=)

In [2]:
# Submit your automl run

### YOUR CODE HERE ###

In [ ]:
# Retrieve and save your best automl model.

### YOUR CODE HERE ###